In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class, MemoryTracker_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import * # import NumericalFunctions 

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="CalculateMoreVariables", dataName="UpdraftArea",
                                dtype='float32')
MemoryTracker = MemoryTracker_Class()

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res, t_res):
    jobs = {
        ('1km', '5min'): 20,
        ('1km', '3min'): 30,
        ('1km', '1min'): 100,
        ('250m', '1min'): 400,
    }
    return jobs.get((res, t_res))
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
####################################
#FUNCTIONS

In [ ]:
#Variable Loading Functions
def GetVarNames():
    return ['winterp','qc','qi']

def GetInputVariables(inputDataDirectory, timeString, varNames):
    inputDictionary = {varName: CallVariable(ModelData, DataManager, timeString, varName) for varName in varNames}
    return inputDictionary

def LoadData(t):
    print(f'Loading data at {ModelData.time_hrs[t]} LT')
    varNames = GetVarNames()
    timeString = ModelData.timeStrings[t]
    inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString, varNames)
    [w,rc,ri] = (inputDictionary[k] for k in varNames)
    return [w,rc,ri]

In [ ]:
#Calculation Functions

#---Binary Array Calcaultion Function---
def GetCloudArray(rc,ri,w):
    """
    Calculates cloudy binary threshold array
    """
    condition1=(rc+ri>1e-5)
    condition2=(w>0)
    A = condition1&condition2
    return A

#---3D Connected-Component Labeling Function---
from scipy.ndimage import label, generate_binary_structure
from scipy.ndimage import label, generate_binary_structure, maximum_filter
from skimage.segmentation import watershed

def GetLabelArray(A, periodic_yx=(True, False), minCount=3):  # minCount=9 Champouillon et al. 2023
    """
    Identify 3D cloud objects via connected-component labeling, following
    the object identification method of:

        Champouillon, A., C. Rio, and F. Couvreux, 2023: Simulating the
        Transition from Shallow to Deep Convection across Scales: The Role
        of Congestus Clouds. J. Atmos. Sci., 80, 2989-3005,
        https://doi.org/10.1175/JAS-D-23-0027.1

    Champouillon et al. define a cloud as "a set of contiguous cloudy
    cells" and discard objects smaller than 9 cells (their Sec. 4a),
    accounting for domain periodicity at the lateral boundaries. This
    function reproduces that definition using 26-connectivity (face +
    edge + corner adjacency; not specified in the paper) and periodic
    padding along the axes given by `periodic_yx`.
    """
    [periodic_y, periodic_x] = periodic_yx
    structure = generate_binary_structure(3, 3)  # 26-connectivity

    pad_y = (1, 1) if periodic_y else (0, 0)
    pad_x = (1, 1) if periodic_x else (0, 0)

    if periodic_y or periodic_x:
        A_padded = np.pad(A, pad_width=((0, 0), pad_y, pad_x), mode='wrap')
        L_padded, n = label(A_padded, structure=structure)

        y_slice = slice(1, -1) if periodic_y else slice(None)
        x_slice = slice(1, -1) if periodic_x else slice(None)
        L = L_padded[:, y_slice, x_slice]
    else:
        L, n = label(A, structure=structure)

    counts = np.bincount(L.ravel())
    small = counts < minCount
    small[0] = False
    L = np.where(small[L], 0, L)

    # relabel to close gaps left by the size filter, so every remaining
    # id from 1..L.max() corresponds to a real, nonempty object
    L, n_final = label(L > 0, structure=structure)
    counts = np.bincount(L.ravel())   # recompute counts to match the new numbering
    return [L, counts]

#---Cloud Type Identification Algorithm---
from scipy import ndimage
import pandas as pd

def ClassifyClouds(L, zh=ModelData.zh, scheme='kumar_champouillon_hybrid'):
    n_labels = L.max()
    if n_labels == 0:
        return pd.DataFrame(columns=['label', 'top_km', 'base_km', 'depth_km', 'cloud_type'])

    # 3D height field where every cell holds its z-level's height (zh[k]), 
    # so it can be compared cell-by-cell against the labeled array L
    Z = np.broadcast_to(zh[:, None, None], L.shape)
    idx = np.arange(1, n_labels + 1) #not counting 0, which is no cloud

    # max height among each label's cells -> cloud-top height per object
    tops = ndimage.maximum(Z, labels=L, index=idx)
    bases = ndimage.minimum(Z, labels=L, index=idx)
    depths = tops - bases

    cloud_types = [classify_object(t, d, scheme=scheme) for t, d in zip(tops, depths)]

    cloudTypes = pd.DataFrame({
        'label': idx, 'top_km': tops, 'base_km': bases,
        'depth_km': depths, 'cloud_type': cloud_types,
    })
    return cloudTypes

def classify_object(top_km, depth_km, scheme):
    if scheme not in SCHEMES:
        raise ValueError(f"scheme must be one of {list(SCHEMES)}, got '{scheme}'")

    for cloud_type, ranges in SCHEMES[scheme].items():
        top_ok = check_in_range(top_km, ranges['top'], inclusive=(False, True))    # (min, max]
        depth_ok = check_in_range(depth_km, ranges['depth'], inclusive=(True, True))  # [min, max]
        if top_ok and depth_ok:
            return cloud_type
    return 'unclassified'
    
def check_in_range(value, bounds, inclusive=(False, True)):
    """checks if value is in bounds"""
    vmin, vmax = bounds
    min_incl, max_incl = inclusive

    ok_min = vmin is None or (value >= vmin if min_incl else value > vmin)
    ok_max = vmax is None or (value <= vmax if max_incl else value < vmax)
    return ok_min and ok_max
    
# --- Threshold Dictionaries (units: km, matching ModelData.zh) ---
KUMAR_SCHEME = {
    # Kumar et al. (2013)
    # Kumar, V. V., C. Jakob, A. Protat, P. T. May, and L. Davies, 2013: The
    # Four Cumulus Cloud Modes and Their Progression During Rainfall Events:
    # A C-Band Polarimetric Radar Perspective. J. Geophys. Res. Atmos., 118,
    # 8375-8389, https://doi.org/10.1002/jgrd.50640
    'cumulus':      {'top': (1.0, 3.0),   'depth': (None, None)},
    'congestus':    {'top': (3.0, 6.5),   'depth': (None, None)},
    'cumulonimbus': {'top': (6.5, 15.0),  'depth': (None, None)},
    'overshooting': {'top': (15.0, None), 'depth': (None, None)},
}

CHAMPOUILLON_SCHEME = {
    # Champouillon et al. (2023)
    # Champouillon, A., C. Rio, and F. Couvreux, 2023: Simulating the
    # Transition from Shallow to Deep Convection across Scales: The Role
    # of Congestus Clouds. J. Atmos. Sci., 80, 2989-3005,
    # https://doi.org/10.1175/JAS-D-23-0027.1
    'cumulus':      {'top': (None, 3.0), 'depth': (None, None)},
    'congestus':    {'top': (3.0, 6.0),  'depth': (0.5, None)},
    'cumulonimbus': {'top': (6.0, None), 'depth': (0.5, None)},
    'others':       {'top': (3.0, None), 'depth': (None, 0.5)},
}


KUMAR_CHAMPOUILLON_HYBRID_SCHEME = {
    # Kumar et al. (2013)
    # Champouillon et al. (2023)
    'cumulus':      {'top': (1.0, 3.0),  'depth': (None, None)},
    'congestus':    {'top': (3.0, 6.0),  'depth': (0.5, None)},
    'cumulonimbus': {'top': (6.0, 15.0), 'depth': (0.5, None)},
    'overshooting': {'top': (15.0, None), 'depth': (0.5, None)},
}

SCHEMES = {
    'kumar': KUMAR_SCHEME,
    'champouillon': CHAMPOUILLON_SCHEME,
    'kumar_champouillon_hybrid': KUMAR_CHAMPOUILLON_HYBRID_SCHEME,
}

In [ ]:
#Plotting Functions
import matplotlib as mpl
def TestPlot(A, L, cloudTypes):
    plt.rcParams.update({
        'font.size': 14,
        'axes.titlesize': 16,
        'axes.labelsize': 14,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
    })

    L_cmap = mpl.colormaps['gist_ncar'].copy()
    L_cmap.set_bad('lightgrey')

    zLevel = np.abs(ModelData.zh - 3).argmin()
    i = 200
    L_slice = np.ma.masked_where(L == 0, L)

    type_order = ['cumulus', 'congestus', 'cumulonimbus', 'overshooting',
                  'others', 'unclassified']
    type_order = [t for t in type_order if t in cloudTypes['cloud_type'].unique()] \
                 + [t for t in cloudTypes['cloud_type'].unique() if t not in type_order]
    type_to_code = {t: i for i, t in enumerate(type_order)}

    max_label = int(L.max())
    label_to_code = np.full(max_label + 1, -1)
    labels_arr = cloudTypes['label'].values.astype(int)
    codes_arr = cloudTypes['cloud_type'].map(type_to_code).values
    label_to_code[labels_arr] = codes_arr

    type_field = label_to_code[L]
    type_field_masked = np.ma.masked_where(type_field < 0, type_field)

    n_types = len(type_order)
    type_cmap = ListedColormap(plt.cm.tab10.colors[:n_types])
    type_cmap.set_bad('lightgrey')
    type_norm = BoundaryNorm(np.arange(-0.5, n_types + 0.5, 1), type_cmap.N)
    legend_handles = [Patch(facecolor=type_cmap(i), label=t)
                       for i, t in enumerate(type_order)]

    # constrained_layout handles multi-axis shared colorbars correctly;
    # tight_layout() does not -- do not call it below
    fig, axes = plt.subplots(3, 2, figsize=(14, 16), constrained_layout=True)
    (ax1, ax2), (ax3, ax4), (ax5, ax6) = axes

    im1 = ax1.contourf(ModelData.xh, ModelData.yh, A[zLevel])
    ax1.set_ylim(60, 140)
    ax1.set_title(f'A: z = {ModelData.zh[zLevel]:.1f} km')
    ax1.set_xlabel('x (km)'); ax1.set_ylabel('y (km)')

    im2 = ax2.contourf(ModelData.yh, ModelData.zh, A[:, :, i])
    ax2.set_ylim(0, 10)
    ax2.set_title(f'A: x = {ModelData.xh[i]:.1f} km')
    ax2.set_xlabel('y (km)'); ax2.set_ylabel('z (km)')

    fig.colorbar(im1, ax=[ax1, ax2], orientation='horizontal',
                 location='bottom', shrink=0.5, label='cloud mask (A)')

    im3 = ax3.pcolormesh(ModelData.xh, ModelData.yh, L_slice[zLevel],
                          cmap=L_cmap, vmin=1, vmax=L.max())
    ax3.set_ylim(60, 140)
    ax3.set_title(f'L: z = {ModelData.zh[zLevel]:.1f} km')
    ax3.set_xlabel('x (km)'); ax3.set_ylabel('y (km)')

    im4 = ax4.pcolormesh(ModelData.yh, ModelData.zh, L_slice[:, :, i],
                          cmap=L_cmap, vmin=1, vmax=L.max())
    ax4.set_ylim(0, 10)
    ax4.set_title(f'L: x = {ModelData.xh[i]:.1f} km')
    ax4.set_xlabel('y (km)'); ax4.set_ylabel('z (km)')

    cbar34 = fig.colorbar(im3, ax=[ax3, ax4], orientation='horizontal',
                           location='bottom', shrink=0.5, label='object label (L)')
    cbar34.set_ticks(np.linspace(1, L.max(), 6, dtype=int))

    im5 = ax5.pcolormesh(ModelData.xh, ModelData.yh, type_field_masked[zLevel],
                          cmap=type_cmap, norm=type_norm)
    ax5.set_ylim(60, 140)
    ax5.set_title(f'cloud type: z = {ModelData.zh[zLevel]:.1f} km')
    ax5.set_xlabel('x (km)'); ax5.set_ylabel('y (km)')

    im6 = ax6.pcolormesh(ModelData.yh, ModelData.zh, type_field_masked[:, :, i],
                          cmap=type_cmap, norm=type_norm)
    ax6.set_ylim(0, 10)
    ax6.set_title(f'cloud type: x = {ModelData.xh[i]:.1f} km')
    ax6.set_xlabel('y (km)'); ax6.set_ylabel('z (km)')

    fig.legend(handles=legend_handles, loc='outside lower center', ncol=n_types)

def MakeSchemeTable(scheme, schemeName, depth_scale=1000):
    rows = []
    for cloud_type, r in scheme.items():
        top = r.get('top', (None, None))
        d = r.get('depth', (None, None))
        depth = (None if d[0] is None else d[0]*depth_scale,
                 None if d[1] is None else d[1]*depth_scale)
        top_str = "No condition" if top == (None, None) else f"{top[0] or ''}\u2013{top[1] or ''} km".strip('\u2013')
        depth_str = "No condition" if depth == (None, None) else f"{depth[0] or ''}\u2013{depth[1] or ''} m".strip('\u2013')
        rows.append({'Cloud type': cloud_type.capitalize(), 'Top': top_str, 'Depth': depth_str})

    styled = (pd.DataFrame(rows)
              .style.hide(axis='index')
              .set_caption(f"Cloud object classification criteria<br>({schemeName})")
              .set_properties(**{
                  'font-size': '16px',
                  'padding': '10px 20px',
                  'text-align': 'center',}).set_table_styles([
                  {'selector': 'th', 'props': [('font-size', '16px'), ('padding', '10px 20px'), ('text-align', 'center')]},
                  {'selector': 'caption', 'props': [('font-size', '18px'), ('caption-side', 'top'), ('font-weight', 'bold')]},]))
    return styled

In [ ]:
####################################
#CALCULATING

In [ ]:
MakeSchemeTable(SCHEMES['kumar_champouillon_hybrid'], schemeName='Kumar–Champouillon hybrid scheme')

In [ ]:
[w,rc,ri]=LoadData(t=170)
A = GetCloudArray(rc,ri,w)
[L,counts]=GetLabelArray(A)
cloudTypes=ClassifyClouds(L)
cloudTypes

In [ ]:
TestPlot(A,L,cloudTypes)